ARIMA - XRP/USD 
Python 3.11.8

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm # For auto_arima
import plotly.graph_objects as go

1. Data Loading and Preparation

In [3]:
# Download Bitcoin data
df_XRP = yf.download(
    tickers=["XRP-USD"],
    start="2020-01-01",
    end="2025-01-01" 
)

# Basic Cleaning and Selection
df_XRP.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
print(f"Original shape: {df_XRP.shape}")
print('Null Values Before:', df_XRP.isnull().values.sum())
# Forward fill common for financial data if few NaNs exist, or drop
df_XRP.ffill(inplace=True) 
print('Null Values After FFill:', df_XRP.isnull().values.sum())

# Select Target and Set Index
df_XRP.reset_index(inplace=True)
df_XRP['Date'] = pd.to_datetime(df_XRP['Date'], format='%Y-%m-%d')
df_XRP = df_XRP[['Date', 'Close']]
df_XRP.set_index('Date', inplace=True)

# Ensure Daily Frequency
df_XRP = df_XRP.asfreq('D')
# Re-check for NaNs introduced by asfreq
print('Null Values After AsFreq:', df_XRP.isnull().values.sum())
df_XRP.ffill(inplace=True) 
print(f"Shape after preproc: {df_XRP.shape}")
print(f"Frequency of the index: {df_XRP.index.freq}")
print("\nData Head:")
print(df_XRP.head())

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

Original shape: (1827, 5)
Null Values Before: 0
Null Values After FFill: 0
Null Values After AsFreq: 0
Shape after preproc: (1827, 1)
Frequency of the index: <Day>

Data Head:
               Close
Date                
2020-01-01  0.192912
2020-01-02  0.192708
2020-01-03  0.187948
2020-01-04  0.193521
2020-01-05  0.194367


2. Data Splitting

In [4]:
# Define split point (e.g., 80% train, 20% test)
train_size = int(len(df_XRP) * 0.80)
train_data = df_XRP[:train_size]
test_data = df_XRP[train_size:]

print(f"\nTraining Data: {len(train_data)} points ({train_data.index.min()} to {train_data.index.max()})")
print(f"Test Data:     {len(test_data)} points ({test_data.index.min()} to {test_data.index.max()})")

# Define Forecast Horizon
fh = len(test_data)


Training Data: 1461 points (2020-01-01 00:00:00 to 2023-12-31 00:00:00)
Test Data:     366 points (2024-01-01 00:00:00 to 2024-12-31 00:00:00)


3. Stationarity Check

In [5]:
# We expect XRP prices to be non-stationary, requiring differencing (d > 0).
def check_stationarity(timeseries):
    print("\nResults of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)
    if dftest[1] <= 0.05:
        print("=> Conclusion: Data is likely Stationary (reject H0)")
    else:
        print("=> Conclusion: Data is likely Non-Stationary (fail to reject H0)")

print("\n--- Stationarity Check on Training Data ---")
check_stationarity(train_data['Close'])


--- Stationarity Check on Training Data ---

Results of Dickey-Fuller Test:
Test Statistic                   -2.634029
p-value                           0.086166
#Lags Used                       23.000000
Number of Observations Used    1437.000000
Critical Value (1%)              -3.434909
Critical Value (5%)              -2.863553
Critical Value (10%)             -2.567842
dtype: float64
=> Conclusion: Data is likely Non-Stationary (fail to reject H0)


4. ARIMA Model Building (using auto_arima)

In [6]:
print("\n--- Running auto_arima ---")
# adjusted 'm' to m=7 for daily data with weekly seasonal ARIMA (assumption)
auto_model = pm.auto_arima(train_data['Close'],
                           start_p=1, start_q=1,
                           test='adf',        
                           max_p=3, max_q=3,  
                           m=7,               
                           start_P=0, seasonal=True,  
                           d=None,           
                           D=None,            
                           trace=True,        
                           error_action='ignore',
                           suppress_warnings=True,
                           stepwise=True)     

print("\n--- Best Model Found ---")
print(auto_model.summary())

# Extract the best model order found
print(f"\nBest ARIMA Order: {auto_model.order}")
print(f"Best Seasonal Order: {auto_model.seasonal_order}")


--- Running auto_arima ---
Performing stepwise search to minimize aic
 ARIMA(1,1,1)(0,0,1)[7] intercept   : AIC=-5109.608, Time=7.41 sec
 ARIMA(0,1,0)(0,0,0)[7] intercept   : AIC=-5106.461, Time=0.19 sec
 ARIMA(1,1,0)(1,0,0)[7] intercept   : AIC=-5111.520, Time=3.96 sec
 ARIMA(0,1,1)(0,0,1)[7] intercept   : AIC=-5111.566, Time=6.57 sec
 ARIMA(0,1,0)(0,0,0)[7]             : AIC=-5108.389, Time=0.24 sec
 ARIMA(0,1,1)(0,0,0)[7] intercept   : AIC=-5112.963, Time=0.47 sec
 ARIMA(0,1,1)(1,0,0)[7] intercept   : AIC=-5111.485, Time=7.90 sec
 ARIMA(0,1,1)(1,0,1)[7] intercept   : AIC=-5110.122, Time=4.14 sec
 ARIMA(1,1,1)(0,0,0)[7] intercept   : AIC=-5111.015, Time=1.20 sec
 ARIMA(0,1,2)(0,0,0)[7] intercept   : AIC=-5111.051, Time=1.47 sec
 ARIMA(1,1,0)(0,0,0)[7] intercept   : AIC=-5113.010, Time=0.92 sec
 ARIMA(1,1,0)(0,0,1)[7] intercept   : AIC=-5111.600, Time=6.23 sec
 ARIMA(1,1,0)(1,0,1)[7] intercept   : AIC=-5117.515, Time=23.38 sec
 ARIMA(1,1,0)(2,0,1)[7] intercept   : AIC=-5117.777, Time

5. Forecasting on Test Set

In [7]:
# Generate predictions for the forecast horizon (length of test set)
# Use the fitted auto_arima model
predictions_arima = auto_model.predict(n_periods=fh)

# Create a pandas Series for the predictions with the correct index
predictions_arima = pd.Series(predictions_arima, index=test_data.index)

print("\n--- ARIMA Predictions (first 5) ---")
print(predictions_arima.head())


--- ARIMA Predictions (first 5) ---
Date
2024-01-01    0.621962
2024-01-02    0.620656
2024-01-03    0.620936
2024-01-04    0.620109
2024-01-05    0.619484
Freq: D, dtype: float64


6. Model Evaluation

In [11]:
# Calculate metrics
y_true = test_data['Close']
y_pred = predictions_arima

rmse_arima = np.sqrt(mean_squared_error(y_true, y_pred))
mae_arima = mean_absolute_error(y_true, y_pred)
mape_arima = mean_absolute_percentage_error(y_true, y_pred)
r2_arima = r2_score(y_true, y_pred) # R-squared can be negative for poor models

# Print Evaluation Metrics
print("\n--- ARIMA Model Evaluation Metrics (Hold-out Set) ---")
print(f"Root Mean Squared Error (RMSE): {rmse_arima:.4f}")
print(f"Mean Absolute Error (MAE):   {mae_arima:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape_arima:.4%}")
print(f"R-squared (R²):              {r2_arima:.4f}") # R² interpretation depends on context

# Print Metrics for Comparison Table in Thesis
print("\n--- ARIMA Hold-out Metrics (for XRP) ---")
print(f"ARIMA Hold-out RMSE:  {rmse_arima:.4f}")
print(f"ARIMA Hold-out MAE:   {mae_arima:.4f}")
print(f"ARIMA Hold-out MAPE:  {mape_arima:.4%}")
print(f"ARIMA Hold-out R²:    {r2_arima:.4f}")


--- ARIMA Model Evaluation Metrics (Hold-out Set) ---
Root Mean Squared Error (RMSE): 0.5270
Mean Absolute Error (MAE):   0.2368
Mean Absolute Percentage Error (MAPE): 20.5498%
R-squared (R²):              -0.0489

--- ARIMA Hold-out Metrics (for XRP) ---
ARIMA Hold-out RMSE:  0.5270
ARIMA Hold-out MAE:   0.2368
ARIMA Hold-out MAPE:  20.5498%
ARIMA Hold-out R²:    -0.0489


7. Visualization

In [13]:
# Prepare data for plotting
plot_train = train_data.reset_index()
plot_test = test_data.reset_index()
plot_pred = pd.DataFrame({'Date': predictions_arima.index, 'Predictions': predictions_arima.values})

# Create Plotly figure
fig = go.Figure()

# Add traces
fig.add_trace(go.Scatter(x=plot_train['Date'], y=plot_train['Close'],
                         mode='lines', name='Actual Price (Train)',
                         line=dict(color='blue')))
fig.add_trace(go.Scatter(x=plot_test['Date'], y=plot_test['Close'],
                         mode='lines', name='Actual Price (Test)',
                         line=dict(color='green')))
fig.add_trace(go.Scatter(x=plot_pred['Date'], y=plot_pred['Predictions'],
                         mode='lines', name='ARIMA Predictions',
                         line=dict(color='red', dash='dash')))

# Update layout
fig.update_layout(
    title="Bitcoin Price Forecasting using ARIMA",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    legend_title="Legend",
    template="plotly_white"
)

# Show plot
fig.show()

8. Data Split Summary

In [14]:
print("\n--- Data Split Summary ---")
train_start_date = train_data.index.min().strftime('%Y-%m-%d')
train_end_date = train_data.index.max().strftime('%Y-%m-%d')
test_start_date = test_data.index.min().strftime('%Y-%m-%d')
test_end_date = test_data.index.max().strftime('%Y-%m-%d')

print(f"Training Set: {len(train_data)} data points from {train_start_date} to {train_end_date}")
print(f"Test Set:     {len(test_data)} data points from {test_start_date} to {test_end_date}")



--- Data Split Summary ---
Training Set: 1461 data points from 2020-01-01 to 2023-12-31
Test Set:     366 data points from 2024-01-01 to 2024-12-31
